# 02 - Tiền xử lý Dữ liệu
## Data Preprocessing

Mục tiêu:
- Xử lý missing values và duplicates
- Chuẩn hóa văn bản: lowercase, remove punctuation, stopwords, lemmatization
- Ghép các trường thành `combined_text`
- Sinh TF-IDF vector

In [ ]:
import sys
sys.path.insert(0, '..')
import pandas as pd
import numpy as np
import re
import os
import joblib
import warnings
warnings.filterwarnings('ignore')

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Download NLTK resources
for r in ['punkt', 'stopwords', 'wordnet', 'omw-1.4', 'punkt_tab']:
    nltk.download(r, quiet=True)

print("Setup complete!")

## 2.1 Load Raw Data

In [ ]:
khan_df = pd.read_csv('../youtube_khan_academy.csv', low_memory=False, nrows=20000)
coursera_df = pd.read_csv('../courses_en.csv', low_memory=False, nrows=15000)
print(f"Khan: {khan_df.shape}")
print(f"Coursera: {coursera_df.shape}")

## 2.2 Xử lý Missing Values và Duplicates

In [ ]:
# Khan Academy
print("=== KHAN ACADEMY ===")
print(f"Before: {len(khan_df)} rows")
khan_df['title'] = khan_df['title'].fillna('')
khan_df['description'] = khan_df['description'].fillna('')
khan_df['channel_title'] = khan_df['channel_title'].fillna('Unknown')
khan_df = khan_df.drop_duplicates(subset=['title']).reset_index(drop=True)
print(f"After dedup: {len(khan_df)} rows")

print("\n=== COURSERA ===")
print(f"Before: {len(coursera_df)} rows")
coursera_df = coursera_df.rename(columns={'name': 'title', 'content': 'description'})
for col in ['title', 'description', 'category', 'skills', 'what_you_learn']:
    if col not in coursera_df.columns:
        coursera_df[col] = ''
    else:
        coursera_df[col] = coursera_df[col].fillna('')
coursera_df = coursera_df.drop_duplicates(subset=['title']).reset_index(drop=True)
print(f"After dedup: {len(coursera_df)} rows")

## 2.3 Tạo Nhãn Độ khó

In [ ]:
def difficulty_from_fog(fog_score):
    """Gunning Fog Index -> Easy/Medium/Hard"""
    try:
        fog = float(fog_score)
        if fog < 8: return 'Easy'
        elif fog <= 12: return 'Medium'
        else: return 'Hard'
    except: return 'Medium'

def difficulty_from_content(category, content=''):
    """Keyword-based difficulty from Coursera metadata"""
    easy_kw = ['beginner', 'introduction', 'intro', 'basic', 'fundamental', 
               'getting started', 'elementary', 'overview', '101']
    hard_kw = ['advanced', 'expert', 'professional', 'deep', 'master',
               'specialization', 'architecture', 'research', 'graduate']
    combined = (str(category) + ' ' + str(content)).lower()
    if any(k in combined for k in hard_kw): return 'Hard'
    elif any(k in combined for k in easy_kw): return 'Easy'
    return 'Medium'

# Assign labels
if 'desc_gunning_fog' in khan_df.columns:
    khan_df['difficulty'] = pd.to_numeric(khan_df['desc_gunning_fog'], errors='coerce').apply(difficulty_from_fog)
else:
    khan_df['difficulty'] = 'Medium'

coursera_df['difficulty'] = coursera_df.apply(
    lambda r: difficulty_from_content(r.get('category', ''), r.get('description', '')), axis=1
)

print("Khan Academy difficulty distribution:")
print(khan_df['difficulty'].value_counts())
print("\nCoursera difficulty distribution:")
print(coursera_df['difficulty'].value_counts())

## 2.4 Chuẩn hóa Văn bản

In [ ]:
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def normalize_text(text):
    """
    Pipeline chuẩn hóa:
    1. Lowercase
    2. Remove URLs
    3. Remove punctuation & digits
    4. Tokenize
    5. Remove stopwords
    6. Lemmatize
    """
    if not isinstance(text, str) or not text.strip():
        return ""
    
    # Step 1: Lowercase
    text = text.lower()
    
    # Step 2: Remove URLs
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    
    # Step 3: Remove punctuation & digits
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Step 4: Tokenize
    try:
        tokens = word_tokenize(text)
    except:
        tokens = text.split()
    
    # Step 5 & 6: Remove stopwords + Lemmatize
    tokens = [lemmatizer.lemmatize(t) for t in tokens 
              if t not in stop_words and len(t) > 2]
    
    return ' '.join(tokens)

# Test
sample = "Introduction to Calculus: Derivatives and Integrals for Beginners!"
print(f"Original: {sample}")
print(f"Cleaned:  {normalize_text(sample)}")

In [ ]:
%%time
# Apply normalization to Khan Academy
print("Normalizing Khan Academy texts...")
khan_df['raw_text'] = khan_df['title'] + ' ' + khan_df['description']
khan_df['clean_text'] = khan_df['raw_text'].apply(normalize_text)

# Apply normalization to Coursera
print("Normalizing Coursera texts...")
coursera_df['raw_text'] = (coursera_df['title'] + ' ' + coursera_df['description'] + 
                            ' ' + coursera_df.get('what_you_learn', '') + 
                            ' ' + coursera_df.get('skills', ''))
coursera_df['clean_text'] = coursera_df['raw_text'].apply(normalize_text)

# Stats
print(f"\nKhan clean text avg length: {khan_df['clean_text'].str.len().mean():.0f} chars")
print(f"Coursera clean text avg length: {coursera_df['clean_text'].str.len().mean():.0f} chars")

## 2.5 Ghép Dataset

In [ ]:
# Add source and ID
khan_df['source'] = 'khan'
khan_df['id'] = 'khan_' + khan_df.index.astype(str)
khan_df['category'] = khan_df['channel_title'].apply(lambda x: str(x).split('|')[0].strip())
khan_df['view_count'] = pd.to_numeric(khan_df.get('view_count', 0), errors='coerce').fillna(0)
khan_df['like_count'] = pd.to_numeric(khan_df.get('like_count', 0), errors='coerce').fillna(0)

coursera_df['source'] = 'coursera'
coursera_df['id'] = 'coursera_' + coursera_df.index.astype(str)
coursera_df['view_count'] = 0
coursera_df['like_count'] = 0

# Merge
select_cols = ['id', 'title', 'raw_text', 'clean_text', 'category', 
               'difficulty', 'source', 'view_count', 'like_count']

unified_df = pd.concat([
    khan_df[[c for c in select_cols if c in khan_df.columns]],
    coursera_df[[c for c in select_cols if c in coursera_df.columns]]
], ignore_index=True)

# Remove empty texts
unified_df = unified_df[unified_df['clean_text'].str.len() > 10].reset_index(drop=True)

# Encode labels
label_map = {'Easy': 0, 'Medium': 1, 'Hard': 2}
unified_df['difficulty_encoded'] = unified_df['difficulty'].map(label_map).fillna(1).astype(int)

print(f"Unified dataset: {unified_df.shape}")
print(f"\nDifficulty distribution:")
print(unified_df['difficulty'].value_counts())
unified_df.head()

## 2.6 Sinh TF-IDF Vector

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

print("Building TF-IDF matrix...")
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.95,
    sublinear_tf=True
)

tfidf_matrix = vectorizer.fit_transform(unified_df['clean_text'])
print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocab size: {len(vectorizer.vocabulary_):,}")
print(f"Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")

# Top terms
feature_names = vectorizer.get_feature_names_out()
print(f"\nSample features: {feature_names[:20].tolist()}")

In [ ]:
# Save processed data
os.makedirs('../data/processed', exist_ok=True)
os.makedirs('../models', exist_ok=True)

unified_df.to_csv('../data/processed/unified_data.csv', index=False)
joblib.dump(vectorizer, '../models/tfidf_vectorizer.pkl')

from scipy.sparse import save_npz
save_npz('../data/processed/tfidf_matrix.npz', tfidf_matrix)

print("Saved:")
print("  ../data/processed/unified_data.csv")
print("  ../models/tfidf_vectorizer.pkl")
print("  ../data/processed/tfidf_matrix.npz")

## 2.7 Kiểm tra kết quả

In [ ]:
# Text length distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
unified_df['text_len'] = unified_df['clean_text'].str.split().str.len()

for i, src in enumerate(['khan', 'coursera']):
    src_data = unified_df[unified_df['source'] == src]['text_len']
    axes[i].hist(src_data, bins=50, color=f'C{i}', alpha=0.8, edgecolor='white')
    axes[i].set_title(f'{src.title()} - Text Word Count')
    axes[i].axvline(src_data.median(), color='red', linestyle='--', 
                    label=f'Median: {src_data.median():.0f}')
    axes[i].legend()

plt.suptitle('Text Length Distribution After Preprocessing', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../report/images/nb02_text_length.png', dpi=150, bbox_inches='tight')
plt.show()
print("Preprocessing complete!")